# Белковые признаки T6SE: AAC, DPC, QSO, CTDC и CTDT

Блокнот для Google Colab. Откройте **Файл → Загрузить блокнот**, затем выберите
**Среда выполнения → Выполнить все** и загрузите приложенный `t6se-training-data.zip`.
Достаточно CPU; GPU не требуется. Для расчёта используются стандартная библиотека
Python, NumPy и pandas, доступные в Colab. Установка propy3 и BioPython не нужна:
матрицы включены, FASTA читается встроенным парсером с поддержкой переносов строк.

Блокнот создаёт отдельный `protein_features.py`, CSV для каждой группы признаков
и каждого класса, общую таблицу, словарь столбцов и ZIP для скачивания.
Репозиторий можно создать позднее из полученного комплекта.

Источники: [Bastion6](https://doi.org/10.1093/bioinformatics/bty155),
[PyPredT6](https://github.com/survivebycoding/PyPredT6/blob/master/pypredt6.py),
[propy3](https://github.com/MartinThoma/propy3).


## 1. Определения

| Блок | Размер | Смысл |
|---|---:|---|
| AAC | 20 | Число остатков каждого типа / N |
| DPC | 400 | Число упорядоченных соседних пар / (N−1), с перекрытиями |
| QSO | 100 | По 20 частотных и 30 порядковых компонент для двух матриц |
| CTDC | 21 | Доли трёх классов каждого из семи свойств |
| CTDT | 21 | Частоты переходов 1↔2, 1↔3, 2↔3 для каждого свойства |

Для QSO: $\tau_d=\sum_{i=1}^{N-d}dist(s_i,s_{i+d})^2$,
$D=1+\omega\sum_{d=1}^{30}\tau_d$; компоненты равны $f_a/D$ и
$\omega\tau_d/D$. Параметры: **maxlag=30**, **ω=0.1**.
Матрицы: Schneider–Wrede и Grantham; каждая нормируется отдельно.

CTDC реализует пункт задания «seven types of physicochemical properties».
Их имена и состав классов доступны в `GROUPS` ниже. CTDT учитывает оба направления
перехода. Частоты представлены долями; округление выполняется только при показе.
Исходные соглашения и отличия от PyPredT6 сохранены в `SOURCES.md`.


In [ ]:
from pathlib import Path
import io, json, zipfile, hashlib, platform, shutil, sys
import numpy as np
import pandas as pd
from IPython.display import display

# Можно задать путь к уже загруженному ZIP; None открывает выбор файла в Colab.
ARCHIVE_PATH = None
MAXLAG = 30
WEIGHT = 0.1
EXPECTED_COUNTS = {"positive": 138, "negative": 1112}
WORK = Path("bastion6_run")
WORK.mkdir(exist_ok=True)
OUT = WORK / "results"
OUT.mkdir(exist_ok=True)
print("Python:", platform.python_version(), "pandas:", pd.__version__, "NumPy:", np.__version__)


## 2. Создание переиспользуемого Python-модуля

Эта ячейка сохраняет исходный код и матрицы в один импортируемый файл.

In [ ]:
%%writefile bastion6_run/protein_features.py
"""AAC, overlapping DPC, QSO, CTDC and CTDT as defined in Bastion6.

QSO distance data: propy3 (Dongsheng Cao, Yizeng Liang, Martin Thoma),
https://github.com/MartinThoma/propy3, GPL-2.0. See LICENSE and SOURCES.md.
Calculations use full precision, without intermediate rounding.
"""
from collections import Counter
from itertools import product
import math

AA = "ACDEFGHIKLMNPQRSTVWY"
GROUPS = {
    "hydrophobicity": ("RKEDQN", "GASTPHY", "CLVIMFW"),
    "vdw_volume": ("GASTPDC", "NVEQIL", "MHKFRYW"),
    "polarity": ("LIFWCMVY", "PATGS", "HQRKNED"),
    "polarizability": ("GASDT", "CPNVEQIL", "KMHFRYW"),
    "charge": ("KR", "ANCQGHILMFPSTWYV", "DE"),
    "secondary_structure": ("EALMQKRH", "VIYCWFT", "GNPSD"),
    "solvent_accessibility": ("ALFCGIVW", "RKQEND", "MSPTHY"),
}
_DISTANCE_ROWS = {
    'SW': [
        [0.0, 0.112, 0.819, 0.827, 0.54, 0.208, 0.696, 0.407, 0.891, 0.406, 0.379, 0.318, 0.191, 0.372, 1.0, 0.094, 0.22, 0.273, 0.739, 0.552],
        [0.114, 0.0, 0.847, 0.838, 0.437, 0.32, 0.66, 0.304, 0.887, 0.301, 0.277, 0.324, 0.157, 0.341, 1.0, 0.176, 0.233, 0.167, 0.639, 0.457],
        [0.729, 0.742, 0.0, 0.124, 0.924, 0.697, 0.435, 0.847, 0.249, 0.841, 0.819, 0.56, 0.657, 0.584, 0.295, 0.667, 0.649, 0.797, 1.0, 0.836],
        [0.79, 0.788, 0.133, 0.0, 0.932, 0.779, 0.406, 0.86, 0.143, 0.854, 0.83, 0.599, 0.688, 0.598, 0.234, 0.726, 0.682, 0.824, 1.0, 0.837],
        [0.508, 0.405, 0.977, 0.918, 0.0, 0.69, 0.663, 0.128, 0.903, 0.131, 0.169, 0.541, 0.42, 0.459, 1.0, 0.548, 0.499, 0.252, 0.207, 0.179],
        [0.206, 0.312, 0.776, 0.807, 0.727, 0.0, 0.769, 0.592, 0.894, 0.591, 0.557, 0.381, 0.323, 0.467, 1.0, 0.158, 0.272, 0.464, 0.923, 0.728],
        [0.896, 0.836, 0.629, 0.547, 0.907, 1.0, 0.0, 0.848, 0.566, 0.842, 0.825, 0.754, 0.777, 0.716, 0.697, 0.865, 0.834, 0.831, 0.981, 0.821],
        [0.403, 0.296, 0.942, 0.891, 0.134, 0.592, 0.652, 0.0, 0.892, 0.013, 0.057, 0.457, 0.311, 0.383, 1.0, 0.443, 0.396, 0.133, 0.339, 0.213],
        [0.889, 0.871, 0.279, 0.149, 0.957, 0.9, 0.438, 0.899, 0.0, 0.892, 0.871, 0.667, 0.757, 0.639, 0.154, 0.825, 0.759, 0.882, 1.0, 0.848],
        [0.405, 0.296, 0.944, 0.892, 0.139, 0.596, 0.653, 0.013, 0.893, 0.0, 0.062, 0.452, 0.309, 0.376, 1.0, 0.443, 0.397, 0.133, 0.341, 0.205],
        [0.383, 0.276, 0.932, 0.879, 0.182, 0.569, 0.648, 0.058, 0.884, 0.062, 0.0, 0.447, 0.285, 0.372, 1.0, 0.417, 0.358, 0.12, 0.391, 0.255],
        [0.424, 0.425, 0.838, 0.835, 0.766, 0.512, 0.78, 0.615, 0.891, 0.603, 0.588, 0.0, 0.266, 0.175, 1.0, 0.361, 0.368, 0.503, 0.945, 0.641],
        [0.22, 0.179, 0.852, 0.831, 0.515, 0.376, 0.696, 0.363, 0.875, 0.357, 0.326, 0.231, 0.0, 0.228, 1.0, 0.196, 0.161, 0.244, 0.72, 0.481],
        [0.512, 0.462, 0.903, 0.861, 0.671, 0.648, 0.765, 0.532, 0.881, 0.518, 0.505, 0.181, 0.272, 0.0, 1.0, 0.461, 0.389, 0.464, 0.831, 0.522],
        [0.919, 0.905, 0.305, 0.225, 0.977, 0.928, 0.498, 0.929, 0.141, 0.92, 0.908, 0.69, 0.796, 0.668, 0.0, 0.86, 0.808, 0.914, 1.0, 0.859],
        [0.1, 0.185, 0.801, 0.812, 0.622, 0.17, 0.718, 0.478, 0.883, 0.474, 0.44, 0.289, 0.181, 0.358, 1.0, 0.0, 0.174, 0.342, 0.827, 0.615],
        [0.251, 0.261, 0.83, 0.812, 0.604, 0.312, 0.737, 0.455, 0.866, 0.453, 0.403, 0.315, 0.159, 0.322, 1.0, 0.185, 0.0, 0.345, 0.816, 0.596],
        [0.275, 0.165, 0.9, 0.867, 0.269, 0.471, 0.649, 0.135, 0.889, 0.134, 0.12, 0.38, 0.212, 0.339, 1.0, 0.322, 0.305, 0.0, 0.472, 0.31],
        [0.658, 0.56, 1.0, 0.931, 0.196, 0.829, 0.678, 0.305, 0.892, 0.304, 0.344, 0.631, 0.555, 0.538, 0.968, 0.689, 0.638, 0.418, 0.0, 0.204],
        [0.587, 0.478, 1.0, 0.932, 0.202, 0.782, 0.678, 0.23, 0.904, 0.219, 0.268, 0.512, 0.444, 0.404, 0.995, 0.612, 0.557, 0.328, 0.244, 0.0]
    ],
    'Grantham': [
        [0, 195, 126, 107, 113, 60, 86, 94, 106, 96, 84, 111, 27, 91, 112, 99, 58, 64, 148, 112],
        [195, 0, 154, 170, 205, 159, 174, 198, 202, 198, 196, 139, 169, 154, 180, 112, 149, 192, 215, 194],
        [126, 154, 0, 45, 177, 94, 81, 168, 101, 172, 160, 23, 108, 61, 96, 65, 85, 152, 181, 160],
        [107, 170, 45, 0, 140, 98, 40, 134, 56, 138, 126, 42, 93, 29, 54, 80, 65, 121, 152, 122],
        [113, 205, 177, 140, 0, 153, 100, 21, 102, 22, 28, 158, 114, 116, 97, 155, 103, 50, 40, 22],
        [60, 159, 94, 98, 153, 0, 98, 135, 127, 138, 127, 80, 42, 87, 125, 56, 59, 109, 184, 147],
        [86, 174, 81, 40, 100, 98, 0, 94, 32, 99, 87, 68, 77, 24, 29, 89, 47, 84, 115, 83],
        [94, 198, 168, 134, 21, 135, 94, 0, 102, 5, 10, 149, 95, 109, 97, 142, 89, 29, 61, 33],
        [106, 202, 101, 56, 102, 127, 32, 102, 0, 107, 95, 94, 103, 53, 26, 121, 78, 97, 110, 85],
        [96, 198, 172, 138, 22, 138, 99, 5, 107, 0, 15, 153, 98, 113, 102, 145, 92, 32, 61, 36],
        [84, 196, 160, 126, 28, 127, 87, 10, 95, 15, 0, 142, 87, 101, 91, 135, 81, 21, 67, 36],
        [111, 139, 23, 42, 158, 80, 68, 149, 94, 153, 142, 0, 91, 46, 86, 46, 65, 133, 174, 143],
        [27, 169, 108, 93, 114, 42, 77, 95, 103, 98, 87, 91, 0, 76, 103, 74, 38, 68, 147, 110],
        [91, 154, 61, 29, 116, 87, 24, 109, 53, 113, 101, 46, 76, 0, 43, 68, 42, 96, 130, 99],
        [112, 180, 96, 54, 97, 125, 29, 97, 26, 102, 91, 86, 103, 43, 0, 110, 71, 96, 101, 77],
        [99, 112, 65, 80, 155, 56, 89, 142, 121, 145, 135, 46, 74, 68, 110, 0, 58, 124, 177, 144],
        [58, 149, 85, 65, 103, 59, 47, 89, 78, 92, 81, 65, 38, 42, 71, 58, 0, 69, 128, 92],
        [64, 192, 152, 121, 50, 109, 84, 29, 97, 32, 21, 133, 68, 96, 96, 124, 69, 0, 88, 55],
        [148, 215, 181, 152, 40, 184, 115, 61, 110, 61, 67, 174, 147, 130, 101, 177, 128, 88, 0, 37],
        [112, 194, 160, 122, 22, 147, 83, 33, 85, 36, 36, 143, 110, 99, 77, 144, 92, 55, 37, 0]
    ]
}
DISTANCES = {name: {a+b: rows[i][j] for i,a in enumerate(AA) for j,b in enumerate(AA)}
             for name,rows in _DISTANCE_ROWS.items()}


def validate_sequence(sequence):
    """Normalize case/whitespace; reject ambiguous residues, gaps and stops."""
    if not isinstance(sequence, str):
        raise TypeError("Sequence must be a string")
    sequence = "".join(sequence.split()).upper()
    if not sequence:
        raise ValueError("Empty protein sequence")
    invalid = set(sequence) - set(AA)
    if invalid:
        raise ValueError(f"Non-standard amino acids: {sorted(invalid)}")
    return sequence


def aac(sequence):
    """20 normalized residue counts n(a)/N, in [0, 1]."""
    s = validate_sequence(sequence)
    counts = Counter(s)
    return {f"AAC_{a}": counts[a] / len(s) for a in AA}


def dpc(sequence):
    """400 ordered adjacent-pair frequencies n(ab)/(N-1), with overlaps."""
    s = validate_sequence(sequence)
    if len(s) < 2:
        raise ValueError("DPC requires at least two residues")
    counts = Counter(a + b for a, b in zip(s, s[1:]))
    return {f"DPC_{a}{b}": counts[a+b] / (len(s)-1)
            for a, b in product(AA, repeat=2)}


def qso(sequence, maxlag=30, weight=0.1):
    """Two (20+maxlag) vectors: Schneider-Wrede (SW) and Grantham.

    tau(d) = sum_i distance(s[i], s[i+d])**2 (no length normalization).
    denominator = 1 + weight * sum_d tau(d).
    The SW matrix is used in its supplied orientation, without symmetrizing.
    A fixed lag requires N > maxlag; short sequences are rejected.
    """
    s = validate_sequence(sequence)
    if isinstance(maxlag, bool) or not isinstance(maxlag, int) or maxlag < 1:
        raise ValueError("maxlag must be a positive integer")
    if len(s) <= maxlag:
        raise ValueError(f"QSO requires length > maxlag={maxlag}; got {len(s)}")
    if not isinstance(weight, (int, float)) or not math.isfinite(weight) or weight < 0:
        raise ValueError("weight must be finite and nonnegative")
    counts = Counter(s)
    result = {}
    for name, matrix in DISTANCES.items():
        taus = [math.fsum(matrix[a+b]**2 for a, b in zip(s, s[d:]))
                for d in range(1, maxlag+1)]
        denominator = 1.0 + weight * math.fsum(taus)
        result.update({f"QSO_{name}_AAC_{a}": counts[a]/len(s)/denominator for a in AA})
        result.update({f"QSO_{name}_lag_{d:02d}": weight*tau/denominator
                       for d, tau in enumerate(taus, 1)})
    return result


def ctdc(sequence):
    """21 composition descriptors: three classes for each of seven properties."""
    s = validate_sequence(sequence)
    counts = Counter(s)
    return {f"CTDC_{prop}_{i}": sum(counts[a] for a in group)/len(s)
            for prop, groups in GROUPS.items() for i, group in enumerate(groups, 1)}


def ctdt(sequence):
    """21 unordered transitions: (n_AB+n_BA)/(N-1), pairs 12, 13, 23.

    Same-class adjacent pairs are not transitions. This is not a conditional
    probability given the first class; its denominator is all N-1 pairs.
    """
    s = validate_sequence(sequence)
    if len(s) < 2:
        raise ValueError("CTDT requires at least two residues")
    result = {}
    for prop, groups in GROUPS.items():
        mapping = {a: i for i, group in enumerate(groups, 1) for a in group}
        classes = [mapping[a] for a in s]
        counts = Counter(tuple(sorted((a, b))) for a, b in zip(classes, classes[1:]) if a != b)
        result.update({f"CTDT_{prop}_{a}{b}": counts[a, b]/(len(s)-1)
                       for a, b in [(1, 2), (1, 3), (2, 3)]})
    return result


def calculate_features(sequence, maxlag=30, weight=0.1):
    """One stable ordered dictionary of 562 descriptors at maxlag=30."""
    result = {}
    for block in (aac(sequence), dpc(sequence), qso(sequence, maxlag, weight),
                  ctdc(sequence), ctdt(sequence)):
        result.update(block)
    return result


def parse_fasta(text):
    """Read wrapped FASTA, preserving full headers and every record, including last.

    Returns (identifier, description, normalized_sequence); no silent filtering.
    """
    header, parts = None, []
    for line_number, line in enumerate(text.splitlines(), 1):
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            if header is not None:
                yield header.split()[0], header, validate_sequence("".join(parts))
            header, parts = line[1:].strip(), []
            if not header:
                raise ValueError(f"Empty FASTA header at line {line_number}")
        else:
            if header is None:
                raise ValueError(f"Sequence before first FASTA header at line {line_number}")
            parts.append(line)
    if header is not None:
        yield header.split()[0], header, validate_sequence("".join(parts))


In [ ]:
#@title Подготовка пояснений, лицензии и импорт модуля
SOURCES_TEXT = '# Источники и принятые соглашения\n\n- Bastion6, разделы 2.2.1 и 2.2.3, таблица 1: https://doi.org/10.1093/bioinformatics/bty155\n- PyPredT6: https://github.com/survivebycoding/PyPredT6/blob/master/pypredt6.py\n- Матрицы QSO из propy3: https://github.com/MartinThoma/propy3/tree/main/propy/data\n- Справка QSO: https://propy3.readthedocs.io/en/latest/QuasiSequenceOrder.html\n- Учебные данные: https://bastion6.erc.monash.edu/static/download/T6SE_training_data.zip\n- Необязательная альтернатива FASTA-парсеру: https://biopython.org/wiki/SeqIO\n\nРеализация следует формулам статьи. PyPredT6 использован как ориентир структуры\nподсчёта состава; его исходный код не скопирован. Там DPC использует str.count\n(без перекрытий) и деление на длину строки. Здесь учитываются все N-1 соседних\nпар, включая перекрытия; переносы строк не входят в длину. Набор групп PyPredT6\nотличается от семи трёхклассовых свойств Bastion6.\n\nВсе частоты записаны долями, не процентами. Промежуточного округления нет.\nQSO: две матрицы, maxlag=30, weight=0.1, tau — сумма квадратов расстояний.\npropy3 округляет частоты и tau до расчёта QSO, а результат до шести знаков;\nпоэтому побитового совпадения с его API здесь не предполагается.\nМатрицы включены в модуль; во время расчёта сеть не требуется.\nЛаг фиксирован для всех белков; при N <= maxlag расчёт останавливается.\nНеизвестные остатки не удаляются, последовательности не обрезаются.\nCTDC и CTDT используют классы из таблицы 1. CTDT суммирует оба направления\nмежклассового перехода; переходы внутри класса не включены.\nИтог: 20 AAC + 400 DPC + 100 QSO + 21 CTDC + 21 CTDT = 562 признака.\nЭто извлечение признаков; обучение классификатора и эволюционные признаки\nне входят в задачу.\n\nМетки этого проекта: positive=1, negative=0. Это локальное соглашение;\nв PyPredT6 используются иные числовые метки.\nПоле sequence_sha256 позволяет обнаружить повторные последовательности.\nПовторы не удаляются автоматически. Каждая исходная запись сохранена.\n\nМатрицы взяты из propy3 (GPL-2.0); этот комплект распространяется под GPL-2.0,\nполный текст — LICENSE. Это не меняет условия использования исходных данных.\nТочные SHA-256 матриц, входного архива, версий и параметры сохраняются в manifest.json.\n'
LICENSE_TEXT = '                    GNU GENERAL PUBLIC LICENSE\n                       Version 2, June 1991\n\n Copyright (C) 1989, 1991 Free Software Foundation, Inc.,\n 51 Franklin Street, Fifth Floor, Boston, MA 02110-1301 USA\n Everyone is permitted to copy and distribute verbatim copies\n of this license document, but changing it is not allowed.\n\n                            Preamble\n\n  The licenses for most software are designed to take away your\nfreedom to share and change it.  By contrast, the GNU General Public\nLicense is intended to guarantee your freedom to share and change free\nsoftware--to make sure the software is free for all its users.  This\nGeneral Public License applies to most of the Free Software\nFoundation\'s software and to any other program whose authors commit to\nusing it.  (Some other Free Software Foundation software is covered by\nthe GNU Lesser General Public License instead.)  You can apply it to\nyour programs, too.\n\n  When we speak of free software, we are referring to freedom, not\nprice.  Our General Public Licenses are designed to make sure that you\nhave the freedom to distribute copies of free software (and charge for\nthis service if you wish), that you receive source code or can get it\nif you want it, that you can change the software or use pieces of it\nin new free programs; and that you know you can do these things.\n\n  To protect your rights, we need to make restrictions that forbid\nanyone to deny you these rights or to ask you to surrender the rights.\nThese restrictions translate to certain responsibilities for you if you\ndistribute copies of the software, or if you modify it.\n\n  For example, if you distribute copies of such a program, whether\ngratis or for a fee, you must give the recipients all the rights that\nyou have.  You must make sure that they, too, receive or can get the\nsource code.  And you must show them these terms so they know their\nrights.\n\n  We protect your rights with two steps: (1) copyright the software, and\n(2) offer you this license which gives you legal permission to copy,\ndistribute and/or modify the software.\n\n  Also, for each author\'s protection and ours, we want to make certain\nthat everyone understands that there is no warranty for this free\nsoftware.  If the software is modified by someone else and passed on, we\nwant its recipients to know that what they have is not the original, so\nthat any problems introduced by others will not reflect on the original\nauthors\' reputations.\n\n  Finally, any free program is threatened constantly by software\npatents.  We wish to avoid the danger that redistributors of a free\nprogram will individually obtain patent licenses, in effect making the\nprogram proprietary.  To prevent this, we have made it clear that any\npatent must be licensed for everyone\'s free use or not licensed at all.\n\n  The precise terms and conditions for copying, distribution and\nmodification follow.\n\n                    GNU GENERAL PUBLIC LICENSE\n   TERMS AND CONDITIONS FOR COPYING, DISTRIBUTION AND MODIFICATION\n\n  0. This License applies to any program or other work which contains\na notice placed by the copyright holder saying it may be distributed\nunder the terms of this General Public License.  The "Program", below,\nrefers to any such program or work, and a "work based on the Program"\nmeans either the Program or any derivative work under copyright law:\nthat is to say, a work containing the Program or a portion of it,\neither verbatim or with modifications and/or translated into another\nlanguage.  (Hereinafter, translation is included without limitation in\nthe term "modification".)  Each licensee is addressed as "you".\n\nActivities other than copying, distribution and modification are not\ncovered by this License; they are outside its scope.  The act of\nrunning the Program is not restricted, and the output from the Program\nis covered only if its contents constitute a work based on the\nProgram (independent of having been made by running the Program).\nWhether that is true depends on what the Program does.\n\n  1. You may copy and distribute verbatim copies of the Program\'s\nsource code as you receive it, in any medium, provided that you\nconspicuously and appropriately publish on each copy an appropriate\ncopyright notice and disclaimer of warranty; keep intact all the\nnotices that refer to this License and to the absence of any warranty;\nand give any other recipients of the Program a copy of this License\nalong with the Program.\n\nYou may charge a fee for the physical act of transferring a copy, and\nyou may at your option offer warranty protection in exchange for a fee.\n\n  2. You may modify your copy or copies of the Program or any portion\nof it, thus forming a work based on the Program, and copy and\ndistribute such modifications or work under the terms of Section 1\nabove, provided that you also meet all of these conditions:\n\n    a) You must cause the modified files to carry prominent notices\n    stating that you changed the files and the date of any change.\n\n    b) You must cause any work that you distribute or publish, that in\n    whole or in part contains or is derived from the Program or any\n    part thereof, to be licensed as a whole at no charge to all third\n    parties under the terms of this License.\n\n    c) If the modified program normally reads commands interactively\n    when run, you must cause it, when started running for such\n    interactive use in the most ordinary way, to print or display an\n    announcement including an appropriate copyright notice and a\n    notice that there is no warranty (or else, saying that you provide\n    a warranty) and that users may redistribute the program under\n    these conditions, and telling the user how to view a copy of this\n    License.  (Exception: if the Program itself is interactive but\n    does not normally print such an announcement, your work based on\n    the Program is not required to print an announcement.)\n\nThese requirements apply to the modified work as a whole.  If\nidentifiable sections of that work are not derived from the Program,\nand can be reasonably considered independent and separate works in\nthemselves, then this License, and its terms, do not apply to those\nsections when you distribute them as separate works.  But when you\ndistribute the same sections as part of a whole which is a work based\non the Program, the distribution of the whole must be on the terms of\nthis License, whose permissions for other licensees extend to the\nentire whole, and thus to each and every part regardless of who wrote it.\n\nThus, it is not the intent of this section to claim rights or contest\nyour rights to work written entirely by you; rather, the intent is to\nexercise the right to control the distribution of derivative or\ncollective works based on the Program.\n\nIn addition, mere aggregation of another work not based on the Program\nwith the Program (or with a work based on the Program) on a volume of\na storage or distribution medium does not bring the other work under\nthe scope of this License.\n\n  3. You may copy and distribute the Program (or a work based on it,\nunder Section 2) in object code or executable form under the terms of\nSections 1 and 2 above provided that you also do one of the following:\n\n    a) Accompany it with the complete corresponding machine-readable\n    source code, which must be distributed under the terms of Sections\n    1 and 2 above on a medium customarily used for software interchange; or,\n\n    b) Accompany it with a written offer, valid for at least three\n    years, to give any third party, for a charge no more than your\n    cost of physically performing source distribution, a complete\n    machine-readable copy of the corresponding source code, to be\n    distributed under the terms of Sections 1 and 2 above on a medium\n    customarily used for software interchange; or,\n\n    c) Accompany it with the information you received as to the offer\n    to distribute corresponding source code.  (This alternative is\n    allowed only for noncommercial distribution and only if you\n    received the program in object code or executable form with such\n    an offer, in accord with Subsection b above.)\n\nThe source code for a work means the preferred form of the work for\nmaking modifications to it.  For an executable work, complete source\ncode means all the source code for all modules it contains, plus any\nassociated interface definition files, plus the scripts used to\ncontrol compilation and installation of the executable.  However, as a\nspecial exception, the source code distributed need not include\nanything that is normally distributed (in either source or binary\nform) with the major components (compiler, kernel, and so on) of the\noperating system on which the executable runs, unless that component\nitself accompanies the executable.\n\nIf distribution of executable or object code is made by offering\naccess to copy from a designated place, then offering equivalent\naccess to copy the source code from the same place counts as\ndistribution of the source code, even though third parties are not\ncompelled to copy the source along with the object code.\n\n  4. You may not copy, modify, sublicense, or distribute the Program\nexcept as expressly provided under this License.  Any attempt\notherwise to copy, modify, sublicense or distribute the Program is\nvoid, and will automatically terminate your rights under this License.\nHowever, parties who have received copies, or rights, from you under\nthis License will not have their licenses terminated so long as such\nparties remain in full compliance.\n\n  5. You are not required to accept this License, since you have not\nsigned it.  However, nothing else grants you permission to modify or\ndistribute the Program or its derivative works.  These actions are\nprohibited by law if you do not accept this License.  Therefore, by\nmodifying or distributing the Program (or any work based on the\nProgram), you indicate your acceptance of this License to do so, and\nall its terms and conditions for copying, distributing or modifying\nthe Program or works based on it.\n\n  6. Each time you redistribute the Program (or any work based on the\nProgram), the recipient automatically receives a license from the\noriginal licensor to copy, distribute or modify the Program subject to\nthese terms and conditions.  You may not impose any further\nrestrictions on the recipients\' exercise of the rights granted herein.\nYou are not responsible for enforcing compliance by third parties to\nthis License.\n\n  7. If, as a consequence of a court judgment or allegation of patent\ninfringement or for any other reason (not limited to patent issues),\nconditions are imposed on you (whether by court order, agreement or\notherwise) that contradict the conditions of this License, they do not\nexcuse you from the conditions of this License.  If you cannot\ndistribute so as to satisfy simultaneously your obligations under this\nLicense and any other pertinent obligations, then as a consequence you\nmay not distribute the Program at all.  For example, if a patent\nlicense would not permit royalty-free redistribution of the Program by\nall those who receive copies directly or indirectly through you, then\nthe only way you could satisfy both it and this License would be to\nrefrain entirely from distribution of the Program.\n\nIf any portion of this section is held invalid or unenforceable under\nany particular circumstance, the balance of the section is intended to\napply and the section as a whole is intended to apply in other\ncircumstances.\n\nIt is not the purpose of this section to induce you to infringe any\npatents or other property right claims or to contest validity of any\nsuch claims; this section has the sole purpose of protecting the\nintegrity of the free software distribution system, which is\nimplemented by public license practices.  Many people have made\ngenerous contributions to the wide range of software distributed\nthrough that system in reliance on consistent application of that\nsystem; it is up to the author/donor to decide if he or she is willing\nto distribute software through any other system and a licensee cannot\nimpose that choice.\n\nThis section is intended to make thoroughly clear what is believed to\nbe a consequence of the rest of this License.\n\n  8. If the distribution and/or use of the Program is restricted in\ncertain countries either by patents or by copyrighted interfaces, the\noriginal copyright holder who places the Program under this License\nmay add an explicit geographical distribution limitation excluding\nthose countries, so that distribution is permitted only in or among\ncountries not thus excluded.  In such case, this License incorporates\nthe limitation as if written in the body of this License.\n\n  9. The Free Software Foundation may publish revised and/or new versions\nof the General Public License from time to time.  Such new versions will\nbe similar in spirit to the present version, but may differ in detail to\naddress new problems or concerns.\n\nEach version is given a distinguishing version number.  If the Program\nspecifies a version number of this License which applies to it and "any\nlater version", you have the option of following the terms and conditions\neither of that version or of any later version published by the Free\nSoftware Foundation.  If the Program does not specify a version number of\nthis License, you may choose any version ever published by the Free Software\nFoundation.\n\n  10. If you wish to incorporate parts of the Program into other free\nprograms whose distribution conditions are different, write to the author\nto ask for permission.  For software which is copyrighted by the Free\nSoftware Foundation, write to the Free Software Foundation; we sometimes\nmake exceptions for this.  Our decision will be guided by the two goals\nof preserving the free status of all derivatives of our free software and\nof promoting the sharing and reuse of software generally.\n\n                            NO WARRANTY\n\n  11. BECAUSE THE PROGRAM IS LICENSED FREE OF CHARGE, THERE IS NO WARRANTY\nFOR THE PROGRAM, TO THE EXTENT PERMITTED BY APPLICABLE LAW.  EXCEPT WHEN\nOTHERWISE STATED IN WRITING THE COPYRIGHT HOLDERS AND/OR OTHER PARTIES\nPROVIDE THE PROGRAM "AS IS" WITHOUT WARRANTY OF ANY KIND, EITHER EXPRESSED\nOR IMPLIED, INCLUDING, BUT NOT LIMITED TO, THE IMPLIED WARRANTIES OF\nMERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE.  THE ENTIRE RISK AS\nTO THE QUALITY AND PERFORMANCE OF THE PROGRAM IS WITH YOU.  SHOULD THE\nPROGRAM PROVE DEFECTIVE, YOU ASSUME THE COST OF ALL NECESSARY SERVICING,\nREPAIR OR CORRECTION.\n\n  12. IN NO EVENT UNLESS REQUIRED BY APPLICABLE LAW OR AGREED TO IN WRITING\nWILL ANY COPYRIGHT HOLDER, OR ANY OTHER PARTY WHO MAY MODIFY AND/OR\nREDISTRIBUTE THE PROGRAM AS PERMITTED ABOVE, BE LIABLE TO YOU FOR DAMAGES,\nINCLUDING ANY GENERAL, SPECIAL, INCIDENTAL OR CONSEQUENTIAL DAMAGES ARISING\nOUT OF THE USE OR INABILITY TO USE THE PROGRAM (INCLUDING BUT NOT LIMITED\nTO LOSS OF DATA OR DATA BEING RENDERED INACCURATE OR LOSSES SUSTAINED BY\nYOU OR THIRD PARTIES OR A FAILURE OF THE PROGRAM TO OPERATE WITH ANY OTHER\nPROGRAMS), EVEN IF SUCH HOLDER OR OTHER PARTY HAS BEEN ADVISED OF THE\nPOSSIBILITY OF SUCH DAMAGES.\n\n                     END OF TERMS AND CONDITIONS\n\n            How to Apply These Terms to Your New Programs\n\n  If you develop a new program, and you want it to be of the greatest\npossible use to the public, the best way to achieve this is to make it\nfree software which everyone can redistribute and change under these terms.\n\n  To do so, attach the following notices to the program.  It is safest\nto attach them to the start of each source file to most effectively\nconvey the exclusion of warranty; and each file should have at least\nthe "copyright" line and a pointer to where the full notice is found.\n\n    propy3, formerly protpy, is a Python package to compute protein descriptors\n    Copyright (C) 2011-2012 Dongsheng Cao and Yizeng Liang\n    Copyright (C) 2020-2022 Martin Thoma\n\n    This program is free software; you can redistribute it and/or modify\n    it under the terms of the GNU General Public License as published by\n    the Free Software Foundation; in version 2 of the License.\n\n    This program is distributed in the hope that it will be useful,\n    but WITHOUT ANY WARRANTY; without even the implied warranty of\n    MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the\n    GNU General Public License for more details.\n\n    You should have received a copy of the GNU General Public License along\n    with this program; if not, write to the Free Software Foundation, Inc.,\n    51 Franklin Street, Fifth Floor, Boston, MA 02110-1301 USA.\n\nAlso add information on how to contact you by electronic and paper mail.\n\nIf the program is interactive, make it output a short notice like this\nwhen it starts in an interactive mode:\n\n    Gnomovision version 69, Copyright (C) year name of author\n    Gnomovision comes with ABSOLUTELY NO WARRANTY; for details type `show w\'.\n    This is free software, and you are welcome to redistribute it\n    under certain conditions; type `show c\' for details.\n\nThe hypothetical commands `show w\' and `show c\' should show the appropriate\nparts of the General Public License.  Of course, the commands you use may\nbe called something other than `show w\' and `show c\'; they could even be\nmouse-clicks or menu items--whatever suits your program.\n\nYou should also get your employer (if you work as a programmer) or your\nschool, if any, to sign a "copyright disclaimer" for the program, if\nnecessary.  Here is a sample; alter the names:\n\n  Yoyodyne, Inc., hereby disclaims all copyright interest in the program\n  `Gnomovision\' (which makes passes at compilers) written by James Hacker.\n\n  <signature of Ty Coon>, 1 April 1989\n  Ty Coon, President of Vice\n\nThis General Public License does not permit incorporating your program into\nproprietary programs.  If your program is a subroutine library, you may\nconsider it more useful to permit linking proprietary applications with the\nlibrary.  If this is what you want to do, use the GNU Lesser General\nPublic License instead of this License.\n'
(WORK / "SOURCES.md").write_text(SOURCES_TEXT, encoding="utf-8")
(WORK / "LICENSE").write_text(LICENSE_TEXT, encoding="utf-8")
import importlib.util
spec = importlib.util.spec_from_file_location("protein_features", WORK / "protein_features.py")
pf = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pf)
display(pd.DataFrame(pf.GROUPS, index=["class_1", "class_2", "class_3"]).T)


## 3. Проверки формул на контрольных последовательностях

In [ ]:
def run_tests():
    from math import isclose
    assert pf.aac("AaA\nC")["AAC_A"] == 0.75
    assert pf.dpc("AAAA")["DPC_AA"] == 1.0  # три перекрывающиеся пары
    assert pf.dpc("ACAC")["DPC_AC"] == 2/3
    assert pf.dpc("ACAC")["DPC_CA"] == 1/3
    assert list(pf.parse_fasta(">first\nAC\nDE\n>last\nFG"))[-1][2] == "FG"
    assert len(list(pf.parse_fasta(">one\nAA"))) == 1
    for prop, groups in pf.GROUPS.items():
        assert len("".join(groups)) == 20 and set("".join(groups)) == set(pf.AA)
        assert isclose(sum(pf.ctdc(pf.AA)[f"CTDC_{prop}_{i}"] for i in [1,2,3]), 1)
    # Charge: K=1, A=2, D=3. Переходы 12 и 23, каждый с частотой 1/2.
    t = pf.ctdt("KAD")
    assert t["CTDT_charge_12"] == t["CTDT_charge_23"] == 0.5
    assert t["CTDT_charge_13"] == 0
    assert pf.ctdt("KAK")["CTDT_charge_12"] == 1  # оба направления
    assert pf.ctdt("KKK")["CTDT_charge_12"] == 0
    q = pf.qso("A"*31)
    assert q["QSO_SW_AAC_A"] == q["QSO_Grantham_AAC_A"] == 1
    assert all(v == 0 for k,v in q.items() if "_lag_" in k)
    # Независимая скалярная проверка QSO на неоднородном белке и двух лагах.
    s = "ACDAC"; weight = 0.1
    result = pf.qso(s, maxlag=2, weight=weight)
    for name, matrix in pf.DISTANCES.items():
        taus = [sum(matrix[s[i]+s[i+d]]**2 for i in range(len(s)-d)) for d in [1,2]]
        denominator = 1 + weight*sum(taus)
        assert isclose(result[f"QSO_{name}_AAC_A"], 0.4/denominator)
        for d in [1,2]:
            assert isclose(result[f"QSO_{name}_lag_{d:02d}"], weight*taus[d-1]/denominator)
        assert isclose(sum(v for k,v in result.items() if k.startswith(f"QSO_{name}_")), 1)
    assert len(pf.calculate_features(pf.AA*2)) == 562
    for function, value in [(pf.aac,"AX"),(pf.aac,""),(pf.dpc,"A"),
                            (pf.ctdt,"A"),(pf.qso,"A"*30)]:
        try:
            function(value)
        except ValueError:
            pass
        else:
            raise AssertionError(f"Expected rejection: {function.__name__}({value!r})")
    print("Контрольные проверки пройдены.")

run_tests()


## 4. Загрузка учебного архива

Выберите ZIP из задания. Читаются только FASTA с именами `Pos` и `Neg`;
служебные файлы macOS игнорируются. Метки: **positive=1**, **negative=0**.
Архив читается напрямую, без распаковки на диск. Если возникает ошибка состава
или размера выборки, расчёт прекращается с пояснением.


In [ ]:
if ARCHIVE_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(candidates) != 1:
        raise ValueError("Загрузите ровно один ZIP с учебными последовательностями")
    ARCHIVE_PATH = candidates[0]
archive_bytes = Path(ARCHIVE_PATH).read_bytes()
archive_sha256 = hashlib.sha256(archive_bytes).hexdigest()
records = []
input_members = []
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    for member in sorted(archive.infolist(), key=lambda m: m.filename):
        path = Path(member.filename)
        if "__MACOSX" in path.parts or path.name.startswith("._"):
            continue
        if path.suffix.lower() not in {".fasta", ".fa", ".faa"}:
            continue
        lower = path.stem.lower()
        if "_pos_" in lower:
            class_name, label = "positive", 1
        elif "_neg_" in lower:
            class_name, label = "negative", 0
        else:
            raise ValueError(f"Не удалось определить класс файла: {member.filename}")
        raw = archive.read(member)
        input_members.append({"file": member.filename, "sha256": hashlib.sha256(raw).hexdigest()})
        for ordinal, (seq_id, description, sequence) in enumerate(pf.parse_fasta(raw.decode("utf-8-sig")), 1):
            records.append({"record_key": f"{class_name}_{len(records)+1:05d}",
                "sequence_id": seq_id, "description": description, "class": class_name,
                "label": label, "length": len(sequence), "source_file": member.filename,
                "record_in_file": ordinal,
                "sequence_sha256": hashlib.sha256(sequence.encode("ascii")).hexdigest(),
                "sequence": sequence})
records_df = pd.DataFrame(records)
if records_df.empty:
    raise ValueError("В архиве не найдены записи FASTA")
counts = records_df["class"].value_counts().to_dict()
if counts != EXPECTED_COUNTS:
    raise ValueError(f"Ожидались {EXPECTED_COUNTS}, получены {counts}")
if records_df["length"].min() <= MAXLAG:
    raise ValueError("В наборе есть последовательность длины <= MAXLAG")
display(records_df.groupby("class")["length"].agg(["count", "min", "max"]))
print("Повторных последовательностей:", records_df["sequence_sha256"].duplicated().sum())


## 5. Расчёт всех признаков и проверка результатов

In [ ]:
feature_rows = []
for i, record in enumerate(records, 1):
    try:
        feature_rows.append(pf.calculate_features(record["sequence"], MAXLAG, WEIGHT))
    except (ValueError, KeyError) as error:
        raise ValueError(f"Ошибка в {record['source_file']}, {record['sequence_id']}: {error}") from error
    if i % 250 == 0:
        print(f"Обработано {i}/{len(records)}")
features_df = pd.DataFrame(feature_rows)
blocks = {name: [c for c in features_df if c.startswith(name+"_")]
          for name in ["AAC", "DPC", "QSO", "CTDC", "CTDT"]}
expected_dimensions = {"AAC":20, "DPC":400, "QSO":2*(20+MAXLAG), "CTDC":21, "CTDT":21}
assert {k:len(v) for k,v in blocks.items()} == expected_dimensions
assert features_df.shape == (1250, sum(expected_dimensions.values()))
assert np.isfinite(features_df.to_numpy()).all()
assert (features_df.to_numpy() >= 0).all()
for name in ["AAC", "DPC"]:
    assert np.allclose(features_df[blocks[name]].sum(axis=1), 1)
for matrix in pf.DISTANCES:
    cols = [c for c in blocks["QSO"] if c.startswith(f"QSO_{matrix}_")]
    assert np.allclose(features_df[cols].sum(axis=1), 1)
for prop in pf.GROUPS:
    cols = [f"CTDC_{prop}_{i}" for i in [1,2,3]]
    assert np.allclose(features_df[cols].sum(axis=1), 1)
    cols = [f"CTDT_{prop}_{pair}" for pair in ["12", "13", "23"]]
    assert (features_df[cols].sum(axis=1) <= 1+1e-12).all()
META = ["record_key", "sequence_id", "description", "class", "label", "length",
        "source_file", "record_in_file", "sequence_sha256"]
combined = pd.concat([records_df[META], features_df], axis=1)
print(f"Готово: {len(combined)} белков × {features_df.shape[1]} признака.")


## 6. Пример каждого блока для положительного и отрицательного белка

Показаны первые восемь столбцов каждого блока. CSV содержат все столбцы.

In [ ]:
example_indices = [records_df.index[records_df["class"] == c][0]
                   for c in ["positive", "negative"]]
for name, columns in blocks.items():
    print(f"{name}: {len(columns)} признаков")
    display(combined.loc[example_indices, ["sequence_id", "class"] + columns[:8]])


## 7. Выгрузка CSV и воспроизводимость

`all_features.csv` содержит все записи и признаки, `positive_features.csv` и
`negative_features.csv` — классы отдельно. Для каждого блока создаются отдельные
CSV по классам и `examples_<блок>.csv` с двумя демонстрационными белками.
Разделитель — запятая, десятичный знак — точка, кодировка UTF-8 с BOM.
Индекс pandas не записывается. `feature_dictionary.csv` описывает каждый признак.
`manifest.json` содержит параметры, размеры, версии и контрольные суммы.


In [ ]:
def save_csv(frame, name):
    frame.to_csv(OUT / name, index=False, encoding="utf-8-sig", float_format="%.17g")

save_csv(combined, "all_features.csv")
save_csv(records_df[META + ["sequence"]], "sequences.csv")
for class_name in ["positive", "negative"]:
    mask = combined["class"] == class_name
    save_csv(combined.loc[mask], f"{class_name}_features.csv")
    for block, columns in blocks.items():
        save_csv(combined.loc[mask, META+columns], f"{class_name}_{block}.csv")
for block, columns in blocks.items():
    save_csv(combined.loc[example_indices, META+columns], f"examples_{block}.csv")

dictionary = []
for name in features_df.columns:
    parts = name.split("_")
    block = parts[0]
    if block == "AAC":
        meaning = f"Частота остатка {parts[1]}: n/N"
    elif block == "DPC":
        meaning = f"Частота пары {parts[1]} с перекрытиями: n/(N-1)"
    elif block == "QSO":
        matrix, kind, value = parts[1:]
        meaning = (f"{matrix}: f({value})/D" if kind == "AAC"
                   else f"{matrix}: weight*tau({int(value)})/D")
    else:
        property_name = "_".join(parts[1:-1]); group = parts[-1]
        if block == "CTDC":
            residues = pf.GROUPS[property_name][int(group)-1]
            meaning = f"{property_name}: доля класса {group} ({residues})"
        else:
            meaning = f"{property_name}: переход {group[0]}<->{group[1]}, (n_AB+n_BA)/(N-1)"
    dictionary.append({"feature":name, "block":block, "definition":meaning, "unit":"dimensionless"})
save_csv(pd.DataFrame(dictionary), "feature_dictionary.csv")
# Проверка сохранения: обратное чтение не меняет число записей и столбцов.
restored = pd.read_csv(OUT / "all_features.csv")
assert restored.shape == combined.shape
assert np.allclose(restored[features_df.columns], features_df, rtol=1e-12, atol=1e-15)
manifest = {"parameters":{"maxlag":MAXLAG,"weight":WEIGHT,"frequency_scale":"fraction"},
    "class_labels":{"positive":1,"negative":0}, "counts":counts,
    "feature_dimensions":expected_dimensions, "input_archive_sha256":archive_sha256,
    "input_members":input_members,
    "versions":{"python":platform.python_version(),"numpy":np.__version__,"pandas":pd.__version__},
    "distance_matrix_sha256":{k: hashlib.sha256(json.dumps(v,sort_keys=True).encode()).hexdigest()
                              for k,v in pf.DISTANCES.items()},
    "module_sha256":hashlib.sha256((WORK/"protein_features.py").read_bytes()).hexdigest(),
    "csv_sha256":{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.glob("*.csv"))},
    "validation":"formula tests, feature invariants, CSV round-trip passed"}
(OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame([{"file":p.name,"KB":round(p.stat().st_size/1024,1)} for p in sorted(OUT.iterdir())]))


## 8. Скачать комплект

В ZIP входят модуль, лицензия, пояснения и таблицы. Сам блокнот скачайте через **Файл → Скачать → Скачать IPYNB**.

In [ ]:
bundle_path = Path("bastion6_features_bundle.zip").resolve()
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ["protein_features.py", "SOURCES.md", "LICENSE"]:
        archive.write(WORK/name, name)
    for path in sorted(OUT.iterdir()):
        if path.is_file():
            archive.write(path, "results/"+path.name)
print("Комплект:", bundle_path.name)
try:
    from google.colab import files
except ImportError:
    print("Локальный запуск: ZIP сохранён в текущей папке.")
else:
    files.download(str(bundle_path))


## Повторное использование модуля

После распаковки комплекта можно импортировать `calculate_features` из
`protein_features.py`, передать строку последовательности и получить словарь
имён и значений. Отдельно доступны `aac`, `dpc`, `qso`, `ctdc`, `ctdt`.
Для короткой демонстрационной последовательности укажите меньший `maxlag`
явно; для учебного набора оставьте 30. Нельзя объединять в одной матрице
признаки, рассчитанные с разными значениями maxlag или weight.
